In [ ]:
import fitz
#import pytesseract
from PIL import Image
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

In [ ]:
# Load model
processor = TrOCRProcessor.from_pretrained('microsoft/trocr-large-handwritten') # transkribus will be an option tbf
model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-large-handwritten')
# analyse trocr-large-handwritten maybe

In [ ]:

# Convert page of PDF to image
doc = fitz.open('../data/handwritten_persia.pdf')
page = doc[0]
pix = page.get_pixmap(dpi=300)
pix.save('page.png')

In [ ]:
import easyocr
import numpy as np
reader = easyocr.Reader(['en'])

In [ ]:

pil_img = Image.open('page.png').convert("RGB")
img = np.array(pil_img)

detections = reader.readtext(img, text_threshold=0.3)

pil_img = Image.fromarray(img).convert("RGB")


In [ ]:
import numpy as np

def bbox_to_xyxy(bbox):
    return min(p[0] for p in bbox), min(p[1] for p in bbox), max(p[0] for p in bbox), max(p[1] for p in bbox)

def group_into_lines(detections, overlap_threshold=0.5):
    boxes = [bbox_to_xyxy(d[0]) for d in detections]
    lines = []
    used = set()

    for i, (x1, y1, x2, y2) in enumerate(boxes):
        if i in used:
            continue
        line = [i]
        used.add(i)
        for j, (jx1, jy1, jx2, jy2) in enumerate(boxes):
            if j in used:
                continue
            # check vertical overlap
            overlap = min(y2, jy2) - max(y1, jy1)
            height = min(y2 - y1, jy2 - jy1)
            if overlap / height > overlap_threshold:
                line.append(j)
                used.add(j)
        lines.append(sorted(line, key=lambda k: boxes[k][0]))  # sort left-right within line

    # sort lines top-down by average y
    lines.sort(key=lambda line: np.mean([boxes[k][1] for k in line]))
    return lines

lines = group_into_lines(detections)

full_text = []
for line in lines:
    line_text = []
    for idx in line:
        bbox, _, _ = detections[idx]
        x1, y1, x2, y2 = bbox_to_xyxy(bbox)
        crop = pil_img.crop((x1, y1, x2, y2))
        pixel_values = processor(crop, return_tensors="pt").pixel_values
        generated_ids = model.generate(pixel_values)
        text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        line_text.append(text)
    full_text.append(" ".join(line_text))

print("\n".join(full_text))

In [ ]:

# Run OCR
image = Image.open('page.png').convert('RGB')
pixel_values = processor(image, return_tensors='pt').pixel_values
generated_ids = model.generate(pixel_values)
text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

print(text) # ok so probably will need segmentation in advance